In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scorecardpy as sc
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import pickle

/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
train =pd.read_csv("../data/train_woe.csv")
test = pd.read_csv("../data/test_woe.csv")

In [3]:
train_X = train.drop("target", axis=1)
train_y = train["target"]

test_X = test.drop("target", axis=1)

In [4]:
#逻辑回归
log_model = LogisticRegression() 
log_result = log_model.fit(train_X, train_y)

In [5]:
#比对预测值
test["predicted_target"] = log_result.predict_proba(test_X)[:, 1]
test[["predicted_target","target"]].head(10)

,predicted_target,target
0,0.114295,0
1,0.255227,1
2,0.109118,0
3,0.535352,0
4,0.108234,0
5,0.131142,0
6,0.094660,0
7,0.142600,1
8,0.511950,1
9,0.150003,0


In [6]:
#模型评估
auc = roc_auc_score(test["target"], test["predicted_target"])
fpr, tpr, threshold = roc_curve(test["target"], test["predicted_target"])
ks = max(tpr - fpr)
auc,ks

(0.7639748034366949, 0.4010412385768774)

0.4<=KS<=0.7，说明模型的总体表现良好，有较高区分度。AUC也>0.5，但不算特别高。

In [7]:
with open("bins.pkl", "rb") as f:
    bins = pickle.load(f)
with open("good_vars.pkl", "rb") as f:
    good_vars = pickle.load(f)

In [8]:
#生成信用评分卡
card = sc.scorecard(
    bins=bins, 
    model=log_result,
    xcolumns=good_vars, 
    points0=600, 
    odds0=1/20, 
    pdo=20
)
report = pd.DataFrame()
for var_name, var_card in card.items():
    var_card['variable'] = var_name
    report = pd.concat([report, var_card])
report

,variable,bin,points
0,basepoints,NaN,550.0
12,PAY_0,"[-inf,0.0)",2.0
13,PAY_0,"[0.0,1.0)",3.0
14,PAY_0,"[1.0,2.0)",-3.0
15,PAY_0,"[2.0,inf)",-11.0
...,...,...,...
30,AGE,"[-inf,26.0)",-1.0
31,AGE,"[26.0,29.0)",0.0
32,AGE,"[29.0,36.0)",1.0
33,AGE,"[36.0,46.0)",0.0


In [9]:
#对原数据进行打分
raw_data = pd.read_excel('../data/default_credit_card.xls', header=1)
raw_data_copy = raw_data.copy() 

raw_data_copy['score'] = sc.scorecard_ply(raw_data_copy, card, only_total_score=True)
print(f"原始数据分数范围: {raw_data_copy['score'].min()} - {raw_data_copy['score'].max()}")

原始数据分数范围: 450.0 - 645.0


/Users/xinranzhou/anaconda3/lib/python3.11/site-packages/scorecardpy/scorecard.py:353: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dat_score.loc[:,'score'] = card_basepoints + dat_score.sum(axis=1)
